# DeepEval MCP Metrics - Live Dummy MCP Trace Workflow

This notebook evaluates LLM tool-use traces generated from a local dummy MCP-style server. The inputs live in an Excel workbook, the Python runner fills the LLM output and tool-call sheets, and this notebook reads those sheets to run DeepEval MCP metrics.


## 1. Install Dependencies

In [ ]:
%pip install deepeval mcp openai pandas openpyxl

## 2. Imports

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

from deepeval.metrics.mcp.mcp_task_completion import MCPTaskCompletionMetric
from deepeval.metrics.mcp_use_metric.mcp_use_metric import MCPUseMetric
from deepeval.models.llms.azure_model import AzureOpenAIModel
from deepeval.test_case import LLMTestCase, MCPServer, MCPToolCall
from mcp.types import CallToolResult, TextContent, Tool

## 3. Paths

The runner reads the `input_cases` sheet and writes `llm_outputs`, `tool_calls`, and `deepeval_objects` into the same workbook.


In [ ]:
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "scripts").exists() and (ROOT_DIR.parent / "scripts").exists():
    ROOT_DIR = ROOT_DIR.parent

SCRIPT_FILE = ROOT_DIR / "scripts" / "live_dummy_mcp_trace_generator.py"
TRACE_FILE = ROOT_DIR / "New Notebooks" / "Live_Dummy_MCP_Traces.xlsx"

SCRIPT_FILE, TRACE_FILE

## 4. Credentials

The trace generator supports either OpenAI or Azure OpenAI environment variables.

For OpenAI, set:

```python
os.environ["OPENAI_API_KEY"] = "..."
os.environ["OPENAI_MODEL"] = "gpt-5"
```

For Azure OpenAI, set:

```python
os.environ["AZURE_OPENAI_API_KEY"] = "..."
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
os.environ["AZURE_OPENAI_API_VERSION"] = "..."
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "..."
```


In [ ]:
# Option A: OpenAI
# os.environ["OPENAI_API_KEY"] = "PASTE_OPENAI_API_KEY_HERE"
# os.environ["OPENAI_MODEL"] = "gpt-5"

# Option B: Azure OpenAI
# os.environ["AZURE_OPENAI_API_KEY"] = "PASTE_AZURE_OPENAI_API_KEY_HERE"
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://PASTE_RESOURCE_NAME.openai.azure.com"
# os.environ["AZURE_OPENAI_API_VERSION"] = "PASTE_AZURE_OPENAI_API_VERSION_HERE"
# os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "PASTE_AZURE_OPENAI_DEPLOYMENT_NAME_HERE

## 5. Create Or Run The Trace Workbook

Run this cell once to create the starter workbook. Edit the `input_cases` sheet if you want different scenarios, then run the same cell again to generate LLM outputs and tool traces.


In [ ]:
%run "$SCRIPT_FILE"

## 6. Read Generated Sheets

In [ ]:
input_cases = pd.read_excel(TRACE_FILE, sheet_name="input_cases")
tool_definitions = pd.read_excel(TRACE_FILE, sheet_name="tool_definitions")
llm_outputs = pd.read_excel(TRACE_FILE, sheet_name="llm_outputs")
tool_calls = pd.read_excel(TRACE_FILE, sheet_name="tool_calls")
deepeval_objects = pd.read_excel(TRACE_FILE, sheet_name="deepeval_objects")

llm_outputs

## 7. Copy-Paste Ready Tool Definitions

This sheet contains the MCP `Tool(...)` definitions in Python syntax.

In [ ]:
pd.set_option("display.max_colwidth", 3000)

tool_definitions[["tool_name", "tool_object"]]

## 8. Copy-Paste Ready Test Cases

This sheet contains the generated `MCPToolCall(...)` and `LLMTestCase(...)` snippets.

In [ ]:
deepeval_objects[["case_id", "dummy_mcp_server_object", "llm_test_case_object"]]

## 9. MCP Server Definition For DeepEval

In [ ]:
dummy_support_server = MCPServer(
    server_name="dummy_support",
    transport="stdio",
    available_tools=[
        Tool(
            name="get_customer_profile",
            description="Get a customer profile by customer ID.",
            inputSchema={
                "type": "object",
                "properties": {"customer_id": {"type": "string"}},
                "required": ["customer_id"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="get_order_status",
            description="Get order status and order details by order ID.",
            inputSchema={
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="search_policy_docs",
            description="Search support policy documents.",
            inputSchema={
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="calculate_refund",
            description="Calculate refund eligibility for an order.",
            inputSchema={
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "reason": {"type": "string"},
                },
                "required": ["order_id", "reason"],
                "additionalProperties": False,
            },
        ),
        Tool(
            name="create_support_ticket",
            description="Create a support ticket for a customer issue.",
            inputSchema={
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"},
                    "issue_summary": {"type": "string"},
                    "priority": {"type": "string", "enum": ["low", "normal", "high"]},
                },
                "required": ["customer_id", "issue_summary", "priority"],
                "additionalProperties": False,
            },
        ),
    ],
)

## 10. Convert Excel Rows Into DeepEval Test Cases

In [ ]:
def make_call_tool_result(result_json):
    result_dict = json.loads(result_json)
    return CallToolResult(
        content=[TextContent(type="text", text=json.dumps(result_dict, ensure_ascii=False))],
        isError=False,
    )


def make_mcp_tool_call(row):
    return MCPToolCall(
        name=row["tool_name"],
        args=json.loads(row["tool_arguments_json"]),
        result=make_call_tool_result(row["tool_result_json"]),
    )


test_cases = []
for _, output_row in llm_outputs.iterrows():
    case_tool_rows = tool_calls[tool_calls["case_id"] == output_row["case_id"]]
    mcp_tool_calls = [make_mcp_tool_call(tool_row) for _, tool_row in case_tool_rows.iterrows()]

    test_cases.append(
        LLMTestCase(
            input=output_row["user_input"],
            actual_output=output_row["actual_output"],
            expected_outcome=output_row["expected_outcome"],
            mcp_servers=[dummy_support_server],
            mcp_tools_called=mcp_tool_calls,
        )
    )

len(test_cases), test_cases[0]

## 11. Judge Model

In [ ]:
AZURE_OPENAI_API_KEY = "PASTE_AZURE_OPENAI_API_KEY_HERE"
AZURE_OPENAI_ENDPOINT = "https://PASTE_RESOURCE_NAME.openai.azure.com"
AZURE_OPENAI_DEPLOYMENT_NAME = "PASTE_AZURE_OPENAI_DEPLOYMENT_NAME_HERE"
AZURE_OPENAI_MODEL_NAME = "gpt-5"
AZURE_OPENAI_API_VERSION = "PASTE_AZURE_OPENAI_API_VERSION_HERE"

AZURE_OPENAI_COST_PER_INPUT_TOKEN = 0.0
AZURE_OPENAI_COST_PER_OUTPUT_TOKEN = 0.0

judge_model = AzureOpenAIModel(
    model=AZURE_OPENAI_MODEL_NAME,
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    cost_per_input_token=AZURE_OPENAI_COST_PER_INPUT_TOKEN,
    cost_per_output_token=AZURE_OPENAI_COST_PER_OUTPUT_TOKEN,
)

## 12. Run MCPUseMetric

In [ ]:
mcp_use_rows = []

for test_case, (_, output_row) in zip(test_cases, llm_outputs.iterrows()):
    metric = MCPUseMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(test_case)
    mcp_use_rows.append(
        {
            "case_id": output_row["case_id"],
            "metric": "MCPUseMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_use_report = pd.DataFrame(mcp_use_rows)
mcp_use_report

## 13. Run MCPTaskCompletionMetric

In [ ]:
task_completion_rows = []

for test_case, (_, output_row) in zip(test_cases, llm_outputs.iterrows()):
    metric = MCPTaskCompletionMetric(model=judge_model, threshold=0.5, include_reason=True, async_mode=False, verbose_mode=True)
    score = metric.measure(test_case)
    task_completion_rows.append(
        {
            "case_id": output_row["case_id"],
            "metric": "MCPTaskCompletionMetric",
            "score": round(score, 3),
            "success": metric.is_successful(),
            "reason": metric.reason,
        }
    )

mcp_task_completion_report = pd.DataFrame(task_completion_rows)
mcp_task_completion_report

## 14. Final Report

In [ ]:
final_report = pd.concat([mcp_use_report, mcp_task_completion_report], ignore_index=True)
final_report